# Canada Municipal Polygons Processing

**Overview:**

This script processes two shapefiles from Canada Statistics:

1. **Designated Places (DPLs)**
2. **Population Centers**

Source: https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/index2021-eng.cfm?year=21

Canada uses a different approach than the US, relying on population centers and designated places to represent municipal polygons. However, these polygons often overlap. To create an equivalent dataset for analysis, this script removes the overlapping portions from the designated places, resulting in a dataset where designated places are represented as **"Designated Places MINUS overlap with Population Centers"**.

The processed data is then simplified using a specified tolerance and visualized on an interactive Folium map with distinct color coding:
- **Blue**: Designated Places
- **Green**: Population Centers
- **Red**: Difference (Designated Places minus Population Centers)

A custom legend is added to the map, and the final map is saved as an HTML file.

In [4]:
import geopandas as gpd
import folium

# =============================================================================
#                           FILE PATHS AND SETTINGS
# =============================================================================
print("Starting processing...")

# Define the paths to the shapefiles
shapefile_path_dpls = '/home/shares/wwri-wildfire/data/multi_domain_data/raw/boundary_layers/canada_designated_places/ldpl000b21a_e.shp'
shapefile_path_pcenters = '/home/shares/wwri-wildfire/data/multi_domain_data/raw/boundary_layers/canada_population_centers/lpc_000b21a_e.shp'

# Update the output shapefile path to the desired filename
output_shapefile_path = '/home/shares/wwri-wildfire/data/infrastructure/intermediate/road_network/2024/canada_designated_places_equivalent/canada_cdp_equivalent.shp'

output_html_path = '/home/shares/wwri-wildfire/data/infrastructure/intermediate/road_network/2024/canada_designated_places_equivalent/can_designated_places_population_centers_map.html'

# Simplification tolerance value
simplify_tolerance = 0.0005

# Function to simplify geometries in a GeoDataFrame
def simplify_geodataframe(gdf, tolerance):
    print("  Simplifying geometries...")
    try:
        gdf['geometry'] = gdf['geometry'].simplify(tolerance, preserve_topology=True)
        print("  GeoDataFrame geometries simplified successfully.")
    except Exception as e:
        print(f"  Error simplifying geometries: {e}")
        raise

try:
    # ---------------------------
    #     Designated Places
    # ---------------------------
    print("Reading Designated Places shapefile...")
    canada_dpls_gdf = gpd.read_file(shapefile_path_dpls)
    print("Designated Places shapefile read successfully.")
    print("Designated Places Columns:", canada_dpls_gdf.columns.tolist())

    # Ensure CRS is EPSG:5070
    if canada_dpls_gdf.crs != "EPSG:5070":
        print("Converting Designated Places CRS to EPSG:5070...")
        canada_dpls_gdf = canada_dpls_gdf.to_crs("EPSG:5070")

    # Filter for specific provinces (PRUID '59' and '60')
    print("Filtering Designated Places for PRUIDs 59 and 60...")
    subset_dpls_gdf = canada_dpls_gdf[canada_dpls_gdf['PRUID'].isin(['59', '60'])]

    # ---------------------------
    #     Population Centers
    # ---------------------------
    print("Reading Population Centers shapefile...")
    canada_pcenters_gdf = gpd.read_file(shapefile_path_pcenters)
    print("Population Centers shapefile read successfully.")
    print("Population Centers Columns:", canada_pcenters_gdf.columns.tolist())

    # Ensure CRS is EPSG:5070
    if canada_pcenters_gdf.crs != "EPSG:5070":
        print("Converting Population Centers CRS to EPSG:5070...")
        canada_pcenters_gdf = canada_pcenters_gdf.to_crs("EPSG:5070")

    # Filter for specific provinces (PRUID '59' and '60')
    print("Filtering Population Centers for PRUIDs 59 and 60...")
    subset_pcenters_gdf = canada_pcenters_gdf[canada_pcenters_gdf['PRUID'].isin(['59', '60'])]

    # =============================================================================
    #          PERFORM DIFFERENCE OPERATION BEFORE SIMPLIFICATION
    # =============================================================================
    print("Performing difference (Designated Places MINUS overlap with Population Centers)...")
    difference_gdf = gpd.overlay(subset_dpls_gdf, subset_pcenters_gdf, how='difference')
    print("Difference GeoDataFrame created successfully.")

    # Save the difference GeoDataFrame to the new output path
    print(f"Saving Difference GeoDataFrame to {output_shapefile_path}...")
    difference_gdf.to_file(output_shapefile_path)
    print("Difference GeoDataFrame saved successfully.")

    # =============================================================================
    #                       SIMPLIFY ALL GEOFRAMES
    # =============================================================================
    print("Simplifying all GeoDataFrames now...")
    simplify_geodataframe(subset_dpls_gdf, simplify_tolerance)
    simplify_geodataframe(subset_pcenters_gdf, simplify_tolerance)
    simplify_geodataframe(difference_gdf, simplify_tolerance)

except Exception as e:
    print(f"Error processing shapefiles: {e}")
    raise

# =============================================================================
#                           CREATE FOLIUM MAP
# =============================================================================
try:
    print("Calculating map center...")
    combined_centroid_y = (
        subset_dpls_gdf.geometry.centroid.y.mean() +
        subset_pcenters_gdf.geometry.centroid.y.mean()
    ) / 2
    combined_centroid_x = (
        subset_dpls_gdf.geometry.centroid.x.mean() +
        subset_pcenters_gdf.geometry.centroid.x.mean()
    ) / 2

    map_center = [combined_centroid_y, combined_centroid_x]
    print(f"Map center: {map_center}")

    print("Creating Folium map...")
    folium_map = folium.Map(location=map_center, zoom_start=5)

    # ---------------------------
    #  Plot Designated Places
    # ---------------------------
    folium.GeoJson(
        subset_dpls_gdf,
        name='Designated Places',
        style_function=lambda feature: {
            'fillColor': 'blue',
            'color': 'blue',
            'weight': 1,
            'fillOpacity': 0.5,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['DPLNAME'],
            aliases=['Designated Place:'],
            localize=True
        )
    ).add_to(folium_map)
    print("Designated Places added to Folium map.")

    # ---------------------------
    #  Plot Population Centers
    # ---------------------------
    folium.GeoJson(
        subset_pcenters_gdf,
        name='Population Centers',
        style_function=lambda feature: {
            'fillColor': 'green',
            'color': 'green',
            'weight': 1,
            'fillOpacity': 0.5,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['PCNAME'],
            aliases=['Population Center:'],
            localize=True
        )
    ).add_to(folium_map)
    print("Population Centers added to Folium map.")

    # ---------------------------
    #   Plot Difference (red)
    # ---------------------------
    folium.GeoJson(
        difference_gdf,
        name='DPL minus Overlap',
        style_function=lambda feature: {
            'fillColor': 'red',
            'color': 'red',
            'weight': 1,
            'fillOpacity': 0.5,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['DPLNAME'],
            aliases=['DPL minus Overlap:'],
            localize=True
        )
    ).add_to(folium_map)
    print("Difference GeoDataFrame (red) added to Folium map.")

    folium.LayerControl().add_to(folium_map)
    print("Layer control added.")

    legend_html = """
    <div style="
    position: fixed; 
    bottom: 50px; left: 50px; width: 180px; height: 110px; 
    border:2px solid grey; z-index:9999; font-size:14px;
    background-color:white;
    padding: 5px;
    ">
    <b>Legend</b><br>
    <i class="fa fa-square" style="color:blue"></i>&nbsp;Designated Places<br>
    <i class="fa fa-square" style="color:green"></i>&nbsp;Population Centers<br>
    <i class="fa fa-square" style="color:red"></i>&nbsp;DPL minus Overlap
    </div>
    """
    folium_map.get_root().html.add_child(folium.Element(legend_html))
    print("Custom legend added.")

    folium_map.save(output_html_path)
    print(f"HTML leaflet map saved as {output_html_path}.")

except Exception as e:
    print(f"Error creating or saving the HTML leaflet map: {e}")
    raise


Starting processing...
Reading Designated Places shapefile...
Designated Places shapefile read successfully.
Designated Places Columns: ['DPLUID', 'DGUID', 'DPLNAME', 'DPLTYPE', 'LANDAREA', 'PRUID', 'geometry']
Converting Designated Places CRS to EPSG:5070...
Filtering Designated Places for PRUIDs 59 and 60...
Reading Population Centers shapefile...
Population Centers shapefile read successfully.
Population Centers Columns: ['PCUID', 'PCPUID', 'DGUID', 'DGUIDP', 'PCNAME', 'PCTYPE', 'PCCLASS', 'LANDAREA', 'PRUID', 'geometry']
Converting Population Centers CRS to EPSG:5070...
Filtering Population Centers for PRUIDs 59 and 60...
Performing difference (Designated Places MINUS overlap with Population Centers)...
Difference GeoDataFrame created successfully.
Saving Difference GeoDataFrame to /home/shares/wwri-wildfire/data/infrastructure/intermediate/road_network/2024/canada_designated_places_equivalent/canada_cdp_equivalent.shp...
Difference GeoDataFrame saved successfully.
Simplifying all 

/home/farnisa/domains/domains_env/lib/python3.10/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


  GeoDataFrame geometries simplified successfully.
  Simplifying geometries...
  GeoDataFrame geometries simplified successfully.
  Simplifying geometries...
  GeoDataFrame geometries simplified successfully.
Calculating map center...
Map center: [np.float64(3332115.382763452), np.float64(-1854185.0248448164)]
Creating Folium map...
Designated Places added to Folium map.
Population Centers added to Folium map.
Difference GeoDataFrame (red) added to Folium map.
Layer control added.
Custom legend added.
HTML leaflet map saved as /home/shares/wwri-wildfire/data/infrastructure/intermediate/road_network/2024/canada_designated_places_equivalent/can_designated_places_population_centers_map.html.
